# **M1向け演習：BiomedCLIP を用いた Zero-shot & Few-shot 脳腫瘍分類**

**目的**

本演習では、CLIP を用いた Zero-shot Classification を体験し、以下を理解することを目的とする。

- CLIP が画像とテキストを共通空間に埋め込む仕組みを理解する
- Zero-shot & Few-shot Classification の流れを理解する
- Prompt の違いによって性能が変化することを確認する

1. Google Colab の準備

In [27]:
!pip install -q open_clip_torch

2. ライブラリの読み込み

In [28]:
import os
import json
import torch
import torch.nn.functional as F
from PIL import Image
from collections import defaultdict

import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, Subset

from open_clip import (
    create_model_from_pretrained,
    get_tokenizer
)

3. 乱数固定

In [29]:
import random
import numpy as np

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed_everything()

4. GPU の確認

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


5.データセットの準備

今回は Brain Tumor MRI Dataset を使用します。

データ構造：
```
Testing/

├── glioma
├── meningioma
├── notumor
└── pituitary
```


Google Drive をマウント：

In [31]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


データセットの場所：

以下のURLよりダウンロードできます👇

https://drive.google.com/file/d/1lUJsjPCY8FxJnVgGqPL9q6brGmsWUPCI/view?usp=drive_link

自分のドライブにアップロードする必要があります。

In [32]:
%cd drive/MyDrive

[Errno 2] No such file or directory: 'drive/MyDrive'
/content/drive/MyDrive


In [33]:
!unzip BTMRI.zip

Archive:  BTMRI.zip
replace BTMRI/BTMRI/glioma_tumor/image00.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [34]:
# ==========================================================
# USER SETTINGSEED
# ==========================================================
OBJECT = "BTMRI"

data_dir = "/content/drive/MyDrive/" + OBJECT + '/' + OBJECT

num_workers = 0
context_length = 256
val_batch_size = 256
test_batch_size = 256

print(f"data_dir: {data_dir}")
print(f"num_workers: {num_workers}")
print(f"context_length: {context_length}")
print(f"val_batch_size: {val_batch_size}")
print(f"test_batch_size: {test_batch_size}")

data_dir: /content/drive/MyDrive/BTMRI/BTMRI
num_workers: 0
context_length: 256
val_batch_size: 256
test_batch_size: 256


6. BiomedCLIP の読み込み

In [35]:
model_name = (
    "hf-hub:microsoft/"
    "BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

tokenizer = get_tokenizer(model_name)

model, preprocess = create_model_from_pretrained(
    model_name
)

model = model.to(device)

print("Model loaded!")

Model loaded!


7. データローダー作成

In [36]:
dataset = datasets.ImageFolder(
    data_dir,
    transform=preprocess
)

loader = DataLoader(
    dataset,
    shuffle=False,
    batch_size=val_batch_size,
    num_workers=num_workers
)

classes = dataset.classes

print(classes)

['glioma_tumor', 'meningioma_tumor', 'normal_brain', 'pituitary_tumor']


8. Prompt を作成

In [37]:
# ===============================================
# Prompt Engineering
# ===============================================
text_prompts = [
    ["this is an image of " + cls for cls in classes],
    [cls + " presented in image" for cls in classes],
    ["a photo of " + cls for cls in classes],
]

9. Zero-shot 推論関数

In [38]:
def evaluate(
    model,
    dataloader,
    text_tokens
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        text_features = model.encode_text(
            text_tokens
        )

        text_features = F.normalize(
            text_features,
            dim=-1
        )

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            image_features = model.encode_image(
                images
            )

            image_features = F.normalize(
                image_features,
                dim=-1
            )

            logits = (
                model.logit_scale.exp()
                * image_features
                @ text_features.T
            )

            preds = logits.argmax(dim=1)

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total

10. Zero-shot 分類の実行

In [39]:
for prompt in text_prompts:
    text_tokens = tokenizer(
        prompt,
        context_length=context_length
    ).to(device)

    acc = evaluate(model, loader, text_tokens)

    print(prompt)
    print(f"Accuracy: {acc:.2%}")

['this is an image of glioma_tumor', 'this is an image of meningioma_tumor', 'this is an image of normal_brain', 'this is an image of pituitary_tumor']
Accuracy: 60.00%
['glioma_tumor presented in image', 'meningioma_tumor presented in image', 'normal_brain presented in image', 'pituitary_tumor presented in image']
Accuracy: 63.87%
['a photo of glioma_tumor', 'a photo of meningioma_tumor', 'a photo of normal_brain', 'a photo of pituitary_tumor']
Accuracy: 57.48%


11. Few-shot Dataset

In [40]:
# ==================================================
# {object}専用 Dataset
# ==================================================
class Dataset(Dataset):
    def __init__(self, root_dir, object="BUSI", split="train", transform=None):
        """
        root_dir:
            {object}/
              ├── {object}/
              └── split_{object}.json

        split:
            train / val / test
        """
        self.root_dir = root_dir
        self.image_root = os.path.join(root_dir, object)
        self.transform = transform

        json_path = os.path.join(root_dir, f"split_{object}.json")

        with open(json_path, "r", encoding="utf-8") as f:
            split_info = json.load(f)

        self.samples = split_info[split]

        # class名取得
        self.classes = sorted(list(set([x[2] for x in self.samples])))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rel_path, label, class_name = self.samples[idx]

        img_path = os.path.join(self.image_root, rel_path)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ==================================================
def get_subset_datasets(dataset, samples_per_class):
    """
    各クラスから指定枚数ランダム抽出
    """
    class_indices = defaultdict(list)

    for idx, sample in enumerate(dataset.samples):
        label = sample[1]
        class_indices[label].append(idx)

    selected_indices = []

    for indices in class_indices.values():
        selected = random.sample(
            indices,
            k=min(samples_per_class, len(indices))
        )
        selected_indices.extend(selected)

    return Subset(dataset, selected_indices)

# ==================================================
# Train Loader
# ==================================================
def creat_train_loader(
    root_folder,
    transform=None,
    object="BUSI",
    shot_num=1,
    num_workers=0
):

    train_dataset = Dataset(
        root_dir=root_folder,
        object=object,
        split="train",
        transform=transform
    )

    train_subset = get_subset_datasets(train_dataset, shot_num)

    num_classes = len(train_dataset.classes)
    train_batch_size = shot_num * num_classes

    print(f"Training datasets size: {len(train_subset)}")
    print(f"Number of classes: {num_classes}")
    print(f"train_batch_size: {train_batch_size}")

    train_loader = DataLoader(
        train_subset,
        batch_size=train_batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=num_workers,
        persistent_workers=num_workers > 0
    )

    return train_loader


# ==================================================
# Val/Test Loader
# ==================================================
def creat_val_test_loader(root_folder,
                          object="BUSI",
                          transform=None,
                          val_batch_size=256,
                          test_batch_size=256,
                          num_workers=8):

    val_dataset = Dataset(
        root_dir=root_folder,
        object=object,
        split="val",
        transform=transform
    )

    test_dataset = Dataset(
        root_dir=root_folder,
        object=object,
        split="test",
        transform=transform
    )

    valid_loader = DataLoader(
        val_dataset,
        batch_size=val_batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=num_workers,
        persistent_workers=num_workers > 0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=test_batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=num_workers,
        persistent_workers=num_workers > 0
    )

    print(f"Object: {object}")
    print(f"Val size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")
    print(f"Number of classes: {len(val_dataset.classes)}")

    return valid_loader, test_loader, val_dataset.classes


In [41]:

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    *preprocess.transforms[2:]
])

valid_transform = preprocess

In [42]:
valid_loader, test_loader, classes = creat_val_test_loader(
    root_folder="/content/drive/MyDrive/" + OBJECT,
    transform=valid_transform,
    object=OBJECT,
    val_batch_size=val_batch_size,
    test_batch_size=test_batch_size,
    num_workers=num_workers
)

Object: BTMRI
Val size: 1141
Test size: 1717
Number of classes: 4


12. Linear Probe Model

In [43]:
class LinearProbe(nn.Module):

    def __init__(self, clip_model, num_classes):

        super().__init__()

        self.clip_model = clip_model

        for param in self.clip_model.parameters():
            param.requires_grad = False

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        features = self.clip_model.encode_image(
            x
        )

        features = F.normalize(
            features,
            dim=-1
        )

        logits = self.classifier(
            features
        )

        return logits

13. Train Function

In [44]:
def train(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Few-shot 推論関数

In [45]:
def evaluate_few_shot(
    model,
    loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            preds = logits.argmax(1)

            correct += (
                preds == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total

14. Few-shot Experiment

In [46]:
# ==========================================================
# Few-shot loop
# ==========================================================
for n in range(5):

    shot_num = 2 ** n

    print(f"\n========== {shot_num}-shot ==========")

    train_loader = creat_train_loader(
        root_folder="/content/drive/MyDrive/" + OBJECT,
        transform=train_transform,
        object=OBJECT,
        shot_num=shot_num,
        num_workers=num_workers
    )

    LP_model = LinearProbe(
        model,
        len(classes)
    ).to(device)

    optimizer = torch.optim.Adam(
        LP_model.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    for epoch in range(20):

        loss = train(
            LP_model,
            train_loader,
            optimizer,
            criterion
        )

    acc = evaluate_few_shot(
        LP_model,
        test_loader
    )

    print(
        f"Test Accuracy: {acc:.2%}"
    )


========== 1-shot ==========
Training datasets size: 4
Number of classes: 4
train_batch_size: 4
Test Accuracy: 67.38%

========== 2-shot ==========
Training datasets size: 8
Number of classes: 4
train_batch_size: 8
Test Accuracy: 66.22%

========== 4-shot ==========
Training datasets size: 16
Number of classes: 4
train_batch_size: 16
Test Accuracy: 67.50%

========== 8-shot ==========
Training datasets size: 32
Number of classes: 4
train_batch_size: 32
Test Accuracy: 79.44%

========== 16-shot ==========
Training datasets size: 64
Number of classes: 4
train_batch_size: 64
Test Accuracy: 82.41%
